Load required libraries and test installation

In [ ]:
from pathlib import Path
from scipy import sparse
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy.external as sce
import harmonypy as hm

import scvi

print("scanpy:", sc.__version__)
print("anndata:", ad.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scvi-tools:", scvi.__version__)


In [ ]:
PROJECT_DIR = Path("./cardiac_regen_practical")
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURE_DIR = RESULTS_DIR / "figures"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

mouse_file = DATA_DIR / "3ae58a8e-a64e-44b5-94c6-af55c88e02d3.h5ad"
zebrafish_file = DATA_DIR / "sce_ZF_sampleIntegration_2022-02-02.h5ad"

Load AnnData objects

In [ ]:
mouse = sc.read_h5ad(mouse_file)
zebrafish = sc.read_h5ad(zebrafish_file)
print("loaded datasets")

# store count layer
mouse.layers["counts"] = mouse.raw.X.copy()
zebrafish.layers["counts"] = zebrafish.raw.X.copy()

Inspect the AnnData structure

Before analysing the data, we should inspect the structure of each object

AnnData object strcuture:
adata.X        expression matrix: cells × genes
adata.obs      cell metadata
adata.var      gene metadata
adata.obsm     multi-dimensional cell embeddings
adata.layers   alternative expression matrices
adata.uns      unstructured analysis results

In [ ]:
# Inspect object dimensions
print("Mouse shape:", mouse.shape)
print("Zebrafish shape:", zebrafish.shape)

print("\nMouse AnnData object:")
print(mouse)

print("\nZebrafish AnnData object:")
print(zebrafish)

# Inspect cell metadata columns
print("Mouse obs columns:")
print(mouse.obs.columns.tolist())

print("\nZebrafish obs columns:")
print(zebrafish.obs.columns.tolist())

# Inspect gene metadata columns
print("Mouse var columns:")
print(mouse.var.columns.tolist())

print("\nZebrafish var columns:")
print(zebrafish.var.columns.tolist())

The zebrafish data only has cluster assignments, not cell identities, so these must be assigned

In [ ]:
# load zebrafish reference data
zebrafish_ref = sc.read_h5ad(DATA_DIR/"adata.h5ad")

# Reference-based label transfer requires both datasets to contain the same genes.
zebrafish_ref.var_names = zebrafish_ref.var["Accession"]
shared_genes = zebrafish_ref.var_names.intersection(zebrafish.var_names)

print("Shared genes:", len(shared_genes))

zfish_ref_shared = zebrafish_ref[:, shared_genes].copy()
zfish_query_shared = zebrafish[:, shared_genes].copy()

# downsample the reference to speed up training
max_cells_per_label = 1000

reference_indices = []

for label, group in zfish_ref_shared.obs.groupby("Cell_type"):
    if group.shape[0] > max_cells_per_label:
        chosen = group.sample(max_cells_per_label, random_state=0).index
    else:
        chosen = group.index

    reference_indices.extend(chosen)

zfish_ref_small = zfish_ref_shared[reference_indices].copy()

print(zfish_ref_small.obs["Cell_type"].value_counts())
print(zfish_ref_small)

# preprocess the datasets through Scanpy
def preprocess_for_label_transfer(adata):
    adata = adata.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    return adata

zfish_ref_small = preprocess_for_label_transfer(zfish_ref_small)
zfish_query_shared = preprocess_for_label_transfer(zfish_query_shared)

sc.pp.highly_variable_genes(
    zfish_ref_small,
    n_top_genes=3000,
    flavor="seurat"
)

sc.pp.pca(
    zfish_ref_small,
    use_highly_variable=True,
    n_comps=30
)

sc.pp.neighbors(zfish_ref_small)
sc.tl.umap(zfish_ref_small)

# perform ultra-lightweight reference mapping
sc.tl.ingest(
    zfish_query_shared,
    zfish_ref_small,
    obs="Cell_type"
)

# Store transferred labels
zfish_query_shared.obs["cell_type_reference_transfer"] = (
    zfish_query_shared.obs["Cell_type"].astype(str)
)

zfish_query_shared.obs["cell_type_reference_transfer"].value_counts()

# visualise transferred labels
sc.pl.umap(
    zfish_query_shared,
    color=[
        "cell_type_reference_transfer",
        "cluster_k50",
        "post.surgery",
    ],
    wspace=0.4
)

# Add transferred labels back to the original zebrafish object
zebrafish.obs["cell_type"] = (
    zfish_query_shared.obs["cell_type_reference_transfer"]
    .reindex(zebrafish.obs_names)
)

zebrafish.obs["cell_type"].value_counts(dropna=False)

The mouse and zebrafish AnnData objects use different metadata column names.

A shared set of columns based on the variables we want to integrate is needed:
- `species`
- `sample_id`
- `condition`
- `timepoint`
- `cell_type_label`

In [ ]:
# Add species labels
mouse.obs["species"] = "mouse"
zebrafish.obs["species"] = "zebrafish"

# Create a shared sample identifier
mouse.obs["sample_id"] = mouse.obs["donor_id"].astype(str)
zebrafish.obs["sample_id"] = zebrafish.obs["donor_id"].astype(str)

# Add condition identifier
mouse.obs["condition"] = mouse.obs["surgery"].astype(str)
zebrafish.obs["condition"] = zebrafish.obs["post.surgery"].astype(str)

# Add timepoint covariate
mouse.obs["timepoint"] = mouse.obs["day"].astype(str)
zebrafish.obs["timepoint"] = zebrafish.obs["post.surgery"].astype(str)

# Add cell_type labels
mouse.obs["cell_type_label"] = mouse.obs["cell_type"].astype(str)
zebrafish.obs["cell_type_label"] = zebrafish.obs["cell_type"].astype(str)

# confirm that both objects contain the same core metadata columns
shared_columns = [
    "species",
    "sample_id",
    "condition",
    "timepoint",
    "cell_type_label",
]

print("Mouse metadata:")
display(mouse.obs[shared_columns].head())
mouse.obs[shared_columns] = mouse.obs[shared_columns].astype("category")

print("Zebrafish metadata:")
display(zebrafish.obs[shared_columns].head())
zebrafish.obs[shared_columns] = zebrafish.obs[shared_columns].astype("category")

Perform ortholog mapping and combine AnnData objects into a single object

In [ ]:
# set var_names to gene symbols
mouse.var_names = mouse.var["feature_name"].astype(str).str.strip()
zebrafish.var_names = zebrafish.var["Symbol"].astype(str).str.strip()

mouse.var_names_make_unique()
zebrafish.var_names_make_unique()

# find one-to-one orthologs for cross-species comparison
orthologs = pd.read_csv(DATA_DIR/"mouse_orthos_2026.06.14.tsv", sep="\t", header=1)

mouse_counts = orthologs["Mouse Symbol"].value_counts()
zfish_counts = orthologs["ZFIN Symbol"].value_counts()

one_to_one = orthologs[
    orthologs["Mouse Symbol"].isin(mouse_counts[mouse_counts == 1].index)
    & orthologs["ZFIN Symbol"].isin(zfish_counts[zfish_counts == 1].index)
].copy()

print("All one-to-one ortholog pairs:", orthologs.shape[0])

orthologs_present = one_to_one[
    one_to_one["Mouse Symbol"].isin(mouse.var_names)
    & one_to_one["ZFIN Symbol"].isin(zebrafish.var_names)
].copy()

orthologs_present = orthologs_present.drop_duplicates(
    subset=["Mouse Symbol", "ZFIN Symbol"]
)

print("Ortholog pairs present in both datasets:", orthologs_present.shape[0])

# replace gene names with orthologs
mouse_genes = orthologs_present["Mouse Symbol"].values
zebrafish_genes = orthologs_present["ZFIN Symbol"].values

mouse_orth = mouse[:, mouse_genes].copy()
zebrafish_orth = zebrafish[:, zebrafish_genes].copy()

zebrafish_orth.var["zebrafish_symbol"] = zebrafish_orth.var_names.astype(str)
zebrafish_orth.var["mouse_ortholog"] = mouse_genes
mouse_orth.var["mouse_symbol"] = mouse_orth.var_names.astype(str)

zebrafish_orth.var_names = mouse_genes
zebrafish_orth = zebrafish_orth[:, mouse_orth.var_names].copy()

print("Mouse ortholog object:", mouse_orth.shape)
print("Zebrafish ortholog object:", zebrafish_orth.shape)
print("Gene names identical:", mouse_orth.var_names.equals(zebrafish_orth.var_names))

# keep only shared columns
mouse_orth.obs = mouse_orth.obs[shared_columns].copy()
zebrafish_orth.obs = zebrafish_orth.obs[shared_columns].copy()

# combine datasets 
combined = ad.concat(
    [mouse_orth, zebrafish_orth],
    label="dataset",
    keys=["mouse", "zebrafish"],
    join="inner",
    merge="first",
)

combined.obs_names_make_unique()

Different datasets often have different granularities of cell types (e.g. Lymphocyte vs CD4+ T Cell), so it is helpful to harmonise them for visualisation purposes

In [ ]:
# harmonise obs
cell_type_harmonisation = {
    # Lymphoid
    "B cell": "B cell",
    "B-cells": "B cell",
    "T cell": "T cell",
    "T-cells": "T cell",
    "mature NK T cell": "T / NK cell",
    "natural killer cell": "T / NK cell",

    # Myeloid / immune
    "macrophage": "Macrophage",
    "Macrophages": "Macrophage",
    "Kupffer cell": "Macrophage",
    "monocyte": "Monocyte",
    "Monocytes": "Monocyte",
    "dendritic cell": "Dendritic cell",
    "granulocyte": "Granulocyte / neutrophil",
    "neutrophil": "Granulocyte / neutrophil",
    "Neutrophils": "Granulocyte / neutrophil",

    # Cardiomyocytes
    "cardiac muscle cell": "Cardiomyocyte",
    "Cardiomyocytes (Atrium)": "Cardiomyocyte",
    "Cardiomyocytes (Ventricle)": "Cardiomyocyte",
    "Cardiomyocytes (ttn.2)": "Cardiomyocyte",

    # Endothelial / endocardial
    "endothelial cell": "Endothelial cell",
    "Bl.ves.EC (apnln)": "Endothelial cell",
    "Bl.ves.EC (lyve1)": "Endothelial cell",
    "Bl.ves.EC (plvapb)": "Endothelial cell",
    "endocardial cell": "Endocardial cell",
    "Endocardium (Atrium)": "Endocardial cell",
    "Endocardium (Ventricle)": "Endocardial cell",
    "Endocardium (frzb)": "Endocardial cell",

    # Epicardial
    "Epicardium (Atrium)": "Epicardial cell",
    "Epicardium (Ventricle)": "Epicardial cell",

    # Fibroblast / stromal
    "fibroblast": "Fibroblast",
    "circulating fibrocyte": "Fibroblast",
    "Valve fibroblasts": "Fibroblast",
    "Fibroblasts (cfd)": "Fibroblast",
    "Fibroblasts (col11a1a)": "Fibroblast",
    "Fibroblasts (col12a1a)": "Fibroblast",
    "Fibroblasts (const.)": "Fibroblast",
    "Fibroblasts (cxcl12a)": "Fibroblast",
    "Fibroblasts (mpeg1.1)": "Fibroblast",
    "Fibroblasts (nppc)": "Fibroblast",
    "Fibroblasts (proliferating)": "Fibroblast",
    "Fibroblasts (spock3)": "Fibroblast",

    # Mural / smooth muscle / perivascular
    "pericyte": "Mural / smooth muscle",
    "Perivascular cells": "Mural / smooth muscle",
    "Smooth muscle cells": "Mural / smooth muscle",

    # Neural / glial
    "Schwann cell": "Neural / glial",
    "Myelin cells": "Neural / glial",
    "Neuronal cells": "Neural / glial",

    # Proliferating
    "Proliferating cells": "Proliferating cell",

    # Non-cardiac / likely off-target tissue labels
    "hepatocyte": "Other / non-cardiac",
    "hepatic stellate cell": "Other / non-cardiac",
    "kidney tubule cell": "Other / non-cardiac",
    "exocrine cell": "Other / non-cardiac",
    "pancreatic A cell": "Other / non-cardiac",
    "pancreatic D cell": "Other / non-cardiac",
    "pancreatic PP cell": "Other / non-cardiac",
    "type B pancreatic cell": "Other / non-cardiac",

    # Unknown
    "unknown": "Unknown",
    "Unknown": "Unknown",
}

condition_harmonisation = {
    # sham surgery
    "ctrl": "control",
    "sham": "control",

    # infarc
    "MI": "injury",
    "dpi1": "injury",
    "dpi7": "injury",
}

timepoint_harmonisation = {
    # controls
    "ctrl": "control",
    
    # day 1 after surgery
    "dpi1": "day_1",
    "d01": "day_1",

    # day 7 after surgery
    "dpi7": "day_7",
    "d07": "day_7",

    # day 30 after surgery
    "d30": "day_30",
}

# Apply harmonisation dictionary
combined.obs["cell_type_harmonised"] = (
    combined.obs["cell_type_label"]
    .map(cell_type_harmonisation)
    .fillna("Other")
    .astype("category")
)

combined.obs["condition_harmonised"] = (
    combined.obs["condition"]
    .map(condition_harmonisation)
    .fillna("Other")
    .astype("category")
)

combined.obs["timepoint_harmonised"] = (
    combined.obs["timepoint"]
    .map(timepoint_harmonisation)
    .fillna("Other")
    .astype("category")
)

Examine the structure of the data before integration

In [ ]:
combined_unintegrated = combined.copy()

sc.pp.normalize_total(combined_unintegrated, target_sum=1e4)
sc.pp.log1p(combined_unintegrated)
sc.pp.highly_variable_genes(
    combined_unintegrated, 
    n_top_genes=3000, 
    batch_key="species", 
    flavor="seurat",
)

# PCA on highly variable genes 
sc.pp.pca(combined_unintegrated, use_highly_variable=True, n_comps=50) 
sc.pl.pca_variance_ratio(combined_unintegrated, n_pcs=50, log=True)

# create a baseline UMAP without integration.
sc.pp.neighbors(
    combined_unintegrated, 
    n_neighbors=15, 
    n_pcs=30, 
    use_rep="X_pca",
) 

sc.tl.umap(combined_unintegrated) 
sc.tl.leiden(
    combined_unintegrated, 
    resolution=0.6, 
    key_added="leiden_unintegrated" ,
)

sc.pl.umap(
    combined_unintegrated, 
    color=[ 
        "species", 
        "condition_harmonised", 
        "timepoint_harmonised", 
        "cell_type_harmonised", 
        "leiden_unintegrated", 
    ], 
    wspace=0.4,
)



Harmony adjusts the PCA embeddings to reduce dataset-associated structure.

In [ ]:
combined_harmony = combined_unintegrated.copy()

# Extract PCA matrix
X_pca = np.asarray(combined_harmony.obsm["X_pca"], dtype=np.float64)

print("Input PCA shape:", X_pca.shape)
print("Expected shape: cells × PCs")
print("Number of cells:", combined_harmony.n_obs)

# Run Harmony 
harmony_out = hm.run_harmony(
    data_mat=X_pca,
    meta_data=combined_harmony.obs,
    vars_use=["species"],  # var to integrate over
    theta=2.0,             # mixing strength; higher = stronger species mixing
    lamb=1.0,              # correction penalty; lower = stronger correction, higher = more conservative
    sigma=0.1,             # softness of Harmony clusters; usually keep at 0.1
    nclust=50,             # number of Harmony clusters; higher = finer correction structure
    tau=500,               # protects smaller groups from being overwhelmed by larger groups
    max_iter_harmony=10,
    random_state=0,
)

# Extract integrated representation
Z_corr = np.asarray(harmony_out.Z_corr)

print("Raw Harmony output shape:", Z_corr.shape)

# Harmonypy versions differ in axis orientation so coerce to cells × PCs
if Z_corr.shape[0] == combined_harmony.n_obs:
    X_harmony = Z_corr
elif Z_corr.shape[1] == combined_harmony.n_obs:
    X_harmony = Z_corr.T
else:
    raise ValueError

print("Final Harmony embedding shape:", X_harmony.shape)

# Store corrected PCs
combined_harmony.obsm["X_pca_harmony"] = X_harmony

sc.pp.neighbors(combined_harmony, n_neighbors=15, use_rep="X_pca_harmony") 
sc.tl.umap(combined_harmony) 
sc.tl.leiden(combined_harmony, resolution=0.6, key_added="leiden_harmony")

sc.pl.umap( 
    combined_harmony, 
    color=[ 
        "species", 
        "condition_harmonised", 
        "timepoint_harmonised", 
        "cell_type_harmonised", 
        "leiden_harmony", 
    ], 
    wspace=0.4 
)


scVI is a deep generative model for single-cell count data which models raw counts directly

In [ ]:
# Use the HVGs selected during joint preprocessing 
hvg_genes = combined_unintegrated.var_names[
    combined_unintegrated.var["highly_variable"]
].tolist()

combined_scvi = combined_unintegrated[:, hvg_genes].copy()

# scVI expects raw counts in the default layer 
combined_scvi.X = combined_scvi.layers["counts"].copy()

# Specify the AnnData object for scVI
scvi.model.SCVI.setup_anndata( 
    combined_scvi, 
    layer="counts", 
    batch_key="species",
)

# Train the model
vae = scvi.model.SCVI( 
    combined_scvi, 
    n_latent=20,                # size of latent space; higher = more capacity, lower = stronger compression
    n_layers=2,                 # number of neural network layers; higher = more flexible but more prone to overfitting
    n_hidden=128,               # hidden units per layer; higher = more model capacity
    dropout_rate=0.1,           # dropout regularisation; higher = more regularisation
    gene_likelihood="nb",       # count likelihood; "nb" is negative binomial, "zinb" adds zero inflation
    dispersion="gene-batch",    # how dispersion varies; useful when species/batches differ strongly
    latent_distribution="normal",  # latent prior; "normal" is standard, "ln" can sometimes improve integration
) 

vae.train(max_epochs=30)

# Extract the  latent space
combined_scvi.obsm["X_scVI"] = vae.get_latent_representation()

# compute UMAP from latent embeddings
sc.pp.neighbors(combined_scvi, use_rep="X_scVI", n_neighbors=15)
sc.tl.umap(combined_scvi) 
sc.tl.leiden(combined_scvi, resolution=0.6, key_added="leiden_scvi")

sc.pl.umap(
    combined_scvi, 
    color=[
        "species", 
        "condition_harmonised", 
        "timepoint_harmonised", 
        "cell_type_harmonised", 
        "leiden_scvi", 
    ], 
    wspace=0.4,
)

There is no single correct integration result so each method should be validated based on its performance on the data.

In [ ]:
# Create one object containing all integration embeddings
combined_methods = combined_unintegrated.copy()
combined_methods.obsm["X_Unintegrated"] = combined_methods.obsm["X_pca"].copy()

# Copy Harmony embedding
combined_methods.obsm["X_Harmony"] = combined_harmony[
    combined_methods.obs_names
].obsm["X_pca_harmony"].copy()

# Copy scVI embedding
combined_methods.obsm["X_scVI"] = combined_scvi[
    combined_methods.obs_names
].obsm["X_scVI"].copy()

print(combined_methods.obsm.keys())

# compare methods side by side

embeddings = {
    "Unintegrated": "X_Unintegrated",
    "Harmony": "X_Harmony",
    "scVI": "X_scVI",
}

for color_by in ["species", "cell_type_harmonised", "condition_harmonised", "timepoint_harmonised"]:
    for method, rep in embeddings.items():
        print(f"Computing UMAP for {method}")
    
        sc.pp.neighbors(
            combined_methods,
            use_rep=rep,
            n_neighbors=15,
            key_added=f"neighbors_{method}",
        )
    
        sc.tl.umap(
            combined_methods,
            neighbors_key=f"neighbors_{method}",
            key_added=f"X_umap_{method}",
        )
    
        sc.pl.embedding(
            combined_methods,
            basis=f"umap_{method}",
            color=color_by,
            title=f"{method}: {color_by}",
            frameon=False,
        )


Quantitative metrics can be used to approximate the quality of integration such based on biological conservation and batch correction

In [ ]:
label_species_table = pd.crosstab(
    combined_methods.obs["cell_type_harmonised"],
    combined_methods.obs["species"],
)

display(label_species_table.head(100))

shared_labels = label_species_table.index[
    (label_species_table > 0).sum(axis=1) >= 2
].tolist()

print("Number of labels present in at least two species:", len(shared_labels))
print(shared_labels[:100])


bm = Benchmarker(
    combined_methods,
    batch_key="species",
    label_key="cell_type_harmonised",
    embedding_obsm_keys=embeddings.values(),
     bio_conservation_metrics=BioConservation(
            isolated_labels=False,                
            nmi_ari_cluster_labels_kmeans=True,
            nmi_ari_cluster_labels_leiden=False,    
            silhouette_label=False,               
            clisi_knn=True,
        ),
        batch_correction_metrics=BatchCorrection(
            bras=False,               
            ilisi_knn=True,
            kbet_per_label=False,      
            graph_connectivity=True,
            pcr_comparison=True,
        ),
    n_jobs=-1,
)

bm.benchmark()
bm.plot_results_table()

Pseudobulk samples in order to conduct DEA

In [ ]:
# Create pseudobulk group ID
combined.obs["pseudobulk_id"] = (
    combined.obs["sample_id"].astype(str)
    + "__" + combined.obs["species"].astype(str)
    + "__" + combined.obs["cell_type_harmonised"].astype(str)
    + "__" + combined.obs["condition_harmonised"].astype(str)
)

# Sum raw counts per pseudobulk group
pb = sc.get.aggregate(
    combined,
    by="pseudobulk_id",
    func="sum",
    layer="counts",
)

pb.X = pb.layers["sum"].copy()

meta = pb.obs_names.to_series().str.split("__", expand=True)

pb.obs["sample_id"] = meta[0].values
pb.obs["species"] = meta[1].values
pb.obs["cell_type_harmonised"] = meta[2].values
pb.obs["condition_harmonised"] = meta[3].values

# Count how many cells contributed to each pseudobulk sample
n_cells = combined.obs["pseudobulk_id"].value_counts()
pb.obs["n_cells"] = pb.obs_names.map(n_cells).astype(int)

# Filter weak pseudobulked samples
pb = pb[pb.obs["n_cells"] >= 20].copy()

print(pb)

# compute valid comparisons
summary = (
    pb.obs
    .groupby(
        ["species", 
         "condition_harmonised",
         "cell_type_harmonised",
        ],
        observed=True
    )
    .agg(
        n_pseudobulks=("sample_id", "count"),
        n_samples=("sample_id", "nunique"),
        total_cells=("n_cells", "sum"),
    )
    .reset_index()
)

valid = (
    summary
    .groupby([
        "species", 
    ], observed=True
).filter(lambda x: x["condition_harmonised"].nunique() >= 2)
)

display(valid)

Run DEA with PyDESeq2

In [ ]:
# Choose comparison to test
label_key = "cell_type_harmonised"
cell_type_to_test = "Other / non-cardiac"

species_to_test = "mouse"

condition_a = "injury"
condition_b = "control"


# Subset pseudobulk object
pb_de = pb[
    (pb.obs["species"].astype(str) == species_to_test)
    & (pb.obs["cell_type_harmonised"].astype(str) == cell_type_to_test)
    & (pb.obs["condition_harmonised"].astype(str).isin([condition_a, condition_b]))
].copy()

print(pb_de)
print(pb_de.obs["condition_harmonised"].value_counts())
display(pb_de.obs[
    ["sample_id", "species", "condition_harmonised", "cell_type_harmonised", "n_cells"]
])

# Stop early if no valid comparison

if pb_de.n_obs == 0:
    raise ValueError(
        "pb_de has 0 pseudobulk samples. "
        "Your species/cell type/timepoint/condition selection does not exist in pb."
    )

if pb_de.obs["condition_harmonised"].nunique() < 2:
    raise ValueError(
        "This subset contains fewer than 2 conditions. "
        "Choose a comparison with both injury and control samples."
    )

replicate_counts = pb_de.obs.groupby("condition_harmonised")["sample_id"].nunique()
print("Replicates per condition:")
print(replicate_counts)

if (replicate_counts < 2).any():
    raise ValueError(
        "Need at least 2 biological samples per condition for a meaningful DEA example."
    )

# Extract counts and metadata 
X = pb_de.layers["sum"]

if sparse.issparse(X):
    X = X.toarray()
else:
    X = np.asarray(X)

counts = pd.DataFrame(
    X,
    index=pb_de.obs_names,
    columns=pb_de.var_names,
)

metadata = pb_de.obs[["condition_harmonised"]].copy()

# Ensure exact index match
metadata = metadata.loc[counts.index]

print("Counts shape:", counts.shape)
print("Metadata shape:", metadata.shape)
print("Indices match:", counts.index.equals(metadata.index))

if not counts.index.equals(metadata.index):
    raise ValueError("counts.index and metadata.index do not match.")

# Remove low-count genes
genes_to_keep = counts.columns[counts.sum(axis=0) >= 10]
counts = counts[genes_to_keep]

print("Final count matrix:", counts.shape)

# Run PyDESeq2
dds = DeseqDataSet(
    counts=counts,
    metadata=metadata,
    design="~ condition_harmonised",
)

dds.deseq2()

stat_res = DeseqStats(
    dds,
    contrast=["condition_harmonised", condition_a, condition_b],
)

stat_res.summary()

de_results = stat_res.results_df.copy()

# Filter for significant DE genes
sig_de = de_results[
    (de_results["padj"] < 0.05)
    & (de_results["log2FoldChange"].abs() > 1)
].copy()

sig_de = sig_de.sort_values(
    ["padj", "log2FoldChange"],
    ascending=[True, False]
)

sig_de